# Some examples of prompt engineering with code using Gemini

In this lab, you'll practice a few essential prompting principles in order to write effective prompts for large language models.

This updated version keeps the lab short, but adds a few professional habits inspired by modern prompt engineering tutorials:

1. define a clear persona for the model,
2. structure prompts into task / constraints / data / output format,
3. separate instructions from input data,
4. use few-shot examples when labels must be stable,
5. ask the model to answer only from evidence when factual grounding matters.

Feel free to try out different models! You can find model names in the Vertex AI documentation:
https://cloud.google.com/vertex-ai/generative-ai/docs/learn/model-versions


In [1]:
import vertexai

from vertexai.generative_models import GenerativeModel, ChatSession



In [2]:
def get_chat_response(chat: ChatSession , prompt: str):
    response = chat.send_message(prompt)
    return response.text

In [6]:
model = GenerativeModel("gemini-2.5-flash-lite")
chat = model.start_chat()

## 0. A small reusable prompt template

A good prompt should not only ask a question. It should usually specify the model's role, the task, the constraints, the input data, and the expected output format.

The helper below also separates the input data from the instructions. This is a simple way to reduce ambiguity when user-provided text contains commands, lists, or noisy formatting.


In [ ]:
def build_prompt(persona: str, task: str, input_text: str, constraints: str = "", output_format: str = "") -> str:
    """Build a structured prompt with a persona, task, constraints, input and output format."""
    return f"""
You are {persona}.

## Task
{task}

## Constraints
{constraints if constraints else "No specific constraints."}

## Input data
<input>
{input_text}
</input>

## Expected output
{output_format if output_format else "Answer clearly and concisely."}
""".strip()


## 1. Text Summarization

Summarizing or shortening text is an important operation in text analytics. Before genAI it was a difficult task. Now it is super easy!

In [7]:
review = """
As VFX and special effects take over the traditional filmmaking methods, \
Nolan is among the remaining few directors who still builds grandiose true-to-life sets \
 and reflects cinematic setpieces by filming them instead of digitising them. \
 Oppenheimer is thus a culmination of Nolan's cinematic genius combined with an \
 incredible story that changed the world in more ways than one. It's incredibly intimate \
 and divisive, with the onus of it's justification being put on the audience instead of \
 the narrator.

The cast is just as incredible as you would expect it to be, and the screenplay flows \
naturally, with a breathtaking score that justifiably draws parallels from Hans Zimmer's \
profound work in Interstellar. Nolan balances the intimacy between the characters while \
 simultaneously juxtaposing it with some of the most impactful scenes ever shown on the big \
  screen. The movie's runtime takes it's time in setting up it's pieces, with the finale \
  leaving you utterly spellbound at the sheer magnitude and scale of the events transpiring \
  right in front of your eyes.

Lastly, for those who want their daily dosage of immediate dopamine and faster pacing \
 in the theatre instead of experiencing a meticulously guided journey, you could wait a \
 few more months for yet another copy paste Fast and Furious flick or a Marvel movie laden \
  with green backdrops and fan-service.
"""

### Summarize with a word/sentence/character limit

In a prompt you can also provide constraints. Note, they may not be fulfilled!

In [8]:
prompt = build_prompt(
    persona="an expert movie reviewer who writes concise summaries",
    task="Summarize the movie review for a reader who has not seen the film.",
    input_text=review,
    constraints="Use at most 30 words. Keep only the main opinion and one key reason.",
    output_format="Return one short paragraph."
)

response = get_chat_response(chat, prompt)
print(response)

Nolan's *Oppenheimer* blends an incredible true story with his signature filmmaking style, prioritizing practical effects and a thought-provoking narrative. The film features a stellar cast and score, delivering a captivating and impactful cinematic experience.



## 2. Inferring

### Classification

Text classification is a core operation in text analytics. Again, very easy to do with genAI, because of all the inherent knowledge that has been already ingested.



In [9]:
review = """
It's a pretty good printer, but my first one so I don't have anything to judge it on. Here are some things you need to know- 1. 
It's a kit, and not entirely assembled, so it's nice to have your own tools to put it together, instead of using the ones that come with it. 2. 
The directions are not really that great. The ones I got didn't quite match and the pictures were a little small. 
There are some really good youtube videos, so just look up S3 assembly. 3. Right now, it's really cheap, in fact droping from $279 to what I paid $175, to 149 as of today. 4. 
The prints, with a little tweaking, are just fine. They have a decent surface finish and seem strong enough for my uses. I really like it at $175, but it would be really an awesome printer at $149.
The Voxelab software works great. Although it doesn't have a touch screen, the menus are easy to use with a turn and press knob. With a 300C hot end and direct drive, there's no filment you can't print.
"""

### Sentiment analysis 

We can also do sentiment analysis without providing any examples (zero-shot). You can also provide some examples, then the operation is called few-shot, eg 5-shot if you give 5-examples.

In [10]:
prompt = f"""
What is the sentiment of the following product review,
which is delimited with triple backticks?

Review text: '''{review}'''
"""
response = get_chat_response(chat, prompt)
print(response)

The sentiment is **moderately positive**.

Here's why:

*   **Positive aspects:** The reviewer likes the print quality after tweaking, finds the Voxelab software great, and highlights the printer's versatility due to the hot end and direct drive. They also express enthusiasm about the price.
*   **Negative aspects:** The reviewer notes the printer is a kit requiring assembly with potentially inadequate tools, and criticizes the poor assembly instructions.
*   **Overall:** While there are criticisms, the reviewer ultimately says they "really like it" at the price they paid and express that it would be "awesome" at an even lower price. This suggests a positive overall experience, tempered by initial difficulties with assembly.



In [11]:
prompt = build_prompt(
    persona="a careful customer-review analyst",
    task="Classify the sentiment of the product review.",
    input_text=review,
    constraints="Choose only one label. Do not explain your answer.",
    output_format='Return exactly one word: "positive", "neutral", or "negative".'
)

response = get_chat_response(chat, prompt)
print(response)

positive



## EXERCISE 1:

With your neighbor, create a prompt to receive a list of French sentences and annotate them as A1,...C2 (i.e., very easy - super difficult).

> It is accurate?

> Is it stable?



In [12]:
## YOUR SOLUTION HERE

### Persona effect

Run your CEFR prompt twice:

1. without a persona,
2. with the persona: *You are a French language teacher specialized in CEFR assessment.*

Compare whether the labels become more stable and better justified.


### Few-shot calibration

When labels are subjective or unstable, give the model a few calibrated examples. This is especially useful for classification tasks such as CEFR levels, sentiment labels, or topic categories.

The goal is not to give many examples, but to give enough examples to anchor the expected scale and output format.


In [ ]:
cefr_examples = [
    ("Je m'appelle Léa. J'habite à Genève.", "A1"),
    ("Même si le train était en retard, nous avons réussi à arriver à l'heure.", "B1"),
    ("Il conviendrait d'examiner les implications économiques de cette réforme.", "C1"),
]

sentences_to_classify = [
    "Je voudrais un café, s'il vous plaît.",
    "Bien que les résultats soient encourageants, ils doivent être interprétés avec prudence.",
    "Le chat dort sur le canapé."
]

examples_block = "\n".join(
    [f'Sentence: "{sentence}"\nCEFR level: {level}' for sentence, level in cefr_examples]
)

sentences_block = "\n".join(
    [f'- "{sentence}"' for sentence in sentences_to_classify]
)

prompt = f"""
You are a French language teacher specialized in CEFR assessment.

## Task
Classify each French sentence according to the CEFR scale.

## Calibration examples
{examples_block}

## Sentences to classify
{sentences_block}

## Constraints
- Use only the labels A1, A2, B1, B2, C1, C2.
- Add a short justification for each label.
- Be consistent with the calibration examples.

## Expected output
Return valid JSON only, as a list of objects with keys:
"sentence", "level", "justification".
""".strip()

response = get_chat_response(chat, prompt)
print(response)


## 3. Expanding

### Extract product and company name from customer reviews

In [13]:
prompt = f"""
Identify the following items from the review text:
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Item" and "Brand" as the keys.
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.

Review text: '''{review}'''
"""
response = get_chat_response(chat, prompt)
print(response)

```json
{
  "Item": "printer",
  "Brand": "Voxelab"
}
```


### Doing multiple tasks at once

In [14]:
prompt = build_prompt(
    persona="a data extraction assistant that returns clean JSON",
    task="Extract structured information from the review.",
    input_text=review,
    constraints="Use null if a field is not explicitly present. The anger field must be a boolean.",
    output_format='Return valid JSON only with the keys: "sentiment", "anger", "item", "brand".'
)

response = get_chat_response(chat, prompt)
print(response)

```json
{
  "Sentiment": "positive",
  "Anger": false,
  "Item": "printer",
  "Brand": "Voxelab"
}
```


### Grounded answers: avoid unsupported claims

For factual questions, it is often safer to force the model to use only the provided text.
A useful pattern is: **answer only if the evidence is present; otherwise say that the information is not available.**


In [ ]:
questions = [
    "Does the printer have a touch screen?",
    "Does the review mention Wi-Fi connectivity?"
]

prompt = build_prompt(
    persona="a careful analyst who only answers from the provided evidence",
    task=f"Answer these questions using only the product review: {questions}",
    input_text=review,
    constraints='If the review does not explicitly support the answer, write "not enough information". Include a short evidence quote when possible.',
    output_format='Return valid JSON only, as a list of objects with keys: "question", "answer", "evidence".'
)

response = get_chat_response(chat, prompt)
print(response)


### Topic detection

In [15]:
prompt = f"""
Determine five topics that are being discussed in the \
following text, which is delimited by triple backticks.

Make each item one or two words long.

Format your response as a list of items separated by commas.

Text sample: '''{review}'''
"""
response = get_chat_response(chat, prompt)
print(response)

Printer, Assembly, Price, Prints, Software



In [16]:
response.split(sep=',')

['Printer', ' Assembly', ' Price', ' Prints', ' Software\n']

### Transforming the output

In [ ]:
data_json = { "resturant employees" :[
    {"name":"Mike", "email":"mike@gmail.com"},
    {"name":"Bob", "email":"bob32@gmail.com"},
    {"name":"John", "email":"john87@gmail.com"}
]}

prompt = f"""
Translate the following python dictionary from JSON to an HTML \
table with column headers and title: {data_json}
"""
response = get_chat_response(chat, prompt)
print(response)

In [ ]:
from IPython.display import display, Markdown, Latex, HTML, JSON
display(HTML(response))